# UNI 01 — extract frozen embeddings (train slides)


In [ ]:
DEBUG = False   # True = 100 slides smoke test

In [ ]:
!pip install -q --upgrade timm
!pip install -q imagecodecs huggingface_hub

In [ ]:
import os, numpy as np, pandas as pd
import torch, timm
from tqdm.auto import tqdm
device = torch.device('cuda')
try:
    import openslide; HAVE_OPENSLIDE = True
except Exception:
    HAVE_OPENSLIDE = False
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
except Exception as e:
    print('set HF token manually if needed:', e)
print('timm', timm.__version__, '| openslide', HAVE_OPENSLIDE)

In [ ]:
data_dir = '/kaggle/input/competitions/prostate-cancer-grade-assessment'
RAW      = os.path.join(data_dir, 'train_images')
EMB_DIR  = '/kaggle/working/uni_embeddings'
W_DIR    = '/kaggle/working/uni_weights'
LEVEL = 1; TILE = 256; N_TILES = 36; TILE_BATCH = 32; EMB_DIM = 1024   # UNI ViT-L
os.makedirs(EMB_DIR, exist_ok=True); os.makedirs(W_DIR, exist_ok=True)
df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
if DEBUG: df = df.sample(100, random_state=42).reset_index(drop=True)
print('slides:', len(df))

In [ ]:
def read_slide(path, level=LEVEL):
    if HAVE_OPENSLIDE:
        s = openslide.OpenSlide(path); lv = min(level, s.level_count - 1)
        img = s.read_region((0, 0), lv, s.level_dimensions[lv]).convert('RGB'); s.close()
        return np.asarray(img)
    import tifffile
    with tifffile.TiffFile(path) as tif:
        ser = tif.series[0]
        arr = ser.levels[min(level, len(ser.levels)-1)].asarray() if len(ser.levels) > 1 else ser.asarray()
    arr = np.asarray(arr)
    if arr.ndim == 2: arr = np.stack([arr]*3, -1)
    return arr[..., :3]

def get_tiles(img, mode=0):
    h, w, _ = img.shape
    pad_h = (TILE-h % TILE) % TILE + ((TILE*mode)//2)
    pad_w = (TILE-w % TILE) % TILE + ((TILE*mode)//2)
    img2 = np.pad(img, [[pad_h//2, pad_h-pad_h//2],[pad_w//2, pad_w-pad_w//2],[0,0]], constant_values=255)
    img3 = img2.reshape(img2.shape[0]//TILE, TILE, img2.shape[1]//TILE, TILE, 3)
    img3 = img3.transpose(0,2,1,3,4).reshape(-1, TILE, TILE, 3)
    if len(img3) < N_TILES:
        img3 = np.pad(img3, [[0, N_TILES-len(img3)],[0,0],[0,0],[0,0]], constant_values=255)
    idxs = np.argsort(img3.reshape(img3.shape[0], -1).sum(-1))[:N_TILES]
    return img3[idxs]                                   # (N_TILES, TILE, TILE, 3) RGB, NOT inverted

import torch.nn.functional as F
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def tiles_to_uni(tiles):                                # UNI expects normal H&E, ImageNet norm, 224px
    x = torch.from_numpy(tiles).float().permute(0, 3, 1, 2) / 255.0
    x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)
    return (x - _MEAN) / _STD

In [ ]:
def build_uni():
    return timm.create_model('vit_large_patch16_224', img_size=224, patch_size=16,
                             init_values=1e-5, num_classes=0, dynamic_img_size=True)

def load_uni(local_weights=None):
    m = build_uni()
    if local_weights and os.path.exists(local_weights):           # offline (Kaggle submission)
        sd = torch.load(local_weights, map_location='cpu')
    else:                                                         # online (download once)
        from huggingface_hub import hf_hub_download
        sd = torch.load(hf_hub_download('MahmoodLab/UNI', filename='pytorch_model.bin'),
                        map_location='cpu')
    m.load_state_dict(sd, strict=True)
    for p in m.parameters(): p.requires_grad_(False)
    return m.eval()
uni = load_uni().to(device)
# save weights to output so the offline inference notebook can load them
import shutil
from huggingface_hub import hf_hub_download
shutil.copy(hf_hub_download('MahmoodLab/UNI', filename='pytorch_model.bin'),
            os.path.join(W_DIR, 'uni_pytorch_model.bin'))
print('UNI ready; weights copied for offline use')

In [ ]:
@torch.no_grad()
def embed_slide(image_id, folder):
    tiles = get_tiles(read_slide(os.path.join(folder, f'{image_id}.tiff')), 0)
    x = tiles_to_uni(tiles).to(device)
    out = []
    for i in range(0, len(x), TILE_BATCH):
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(uni(x[i:i+TILE_BATCH]).float().cpu())
    return torch.cat(out).numpy().astype(np.float16)              # (N_TILES, EMB_DIM)

In [ ]:
failed = []
for image_id in tqdm(df.image_id.tolist(), desc='embedding'):
    out = os.path.join(EMB_DIR, f'{image_id}.npy')
    if os.path.exists(out): continue
    try: np.save(out, embed_slide(image_id, RAW))
    except Exception as e: failed.append(image_id); print('FAIL', image_id, e)
pd.read_csv(os.path.join(data_dir, 'train.csv')).to_csv(os.path.join(EMB_DIR, 'train.csv'), index=False)
print('done. embeddings:', len(os.listdir(EMB_DIR)), '| failed:', len(failed))